In [ ]:

# Install Optuna
!pip install -q optuna

# Import Optuna
import optuna

# Verify installation
print("Optuna version:", optuna.__version__)
print("Optuna imported successfully!")


Optuna version: 5.0.0
Optuna imported successfully!


In [9]:
!pip install -q fvcore
from fvcore.nn import FlopCountAnalysis

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
# Install + Upload dataset
!pip install -q albumentations optuna einops timm fvcore huggingface_hub

import os, glob, zipfile
from google.colab import files

ROOT = "/content/data/IDRiD"
os.makedirs(ROOT, exist_ok=True)

print("Select your dataset zip (B__Disease_Grading.zip) ...")
uploaded = files.upload()
for fname in uploaded.keys():
    with zipfile.ZipFile(fname, "r") as z:
        z.extractall(ROOT)
    os.remove(fname)
    print("Extracted:", fname)


csvs = glob.glob(f"{ROOT}/**/*.csv", recursive=True)
imgs = glob.glob(f"{ROOT}/**/*.jpg", recursive=True)
print("CSV files:", *csvs, sep="\n  ")
print("Images:", len(imgs))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Select your dataset zip (B__Disease_Grading.zip) ...


Saving B. Disease Grading.zip to B. Disease Grading.zip
Extracted: B. Disease Grading.zip
CSV files:
  /content/data/IDRiD/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv
  /content/data/IDRiD/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
Images: 516


In [ ]:
#  Locate dataset folders
import os, glob

ROOT = "/content/data/IDRiD"
csv_train = glob.glob(f"{ROOT}/**/*Training Labels*.csv", recursive=True)[0]
csv_test  = glob.glob(f"{ROOT}/**/*Testing Labels*.csv",  recursive=True)[0]
img_train_dir = [d for d in glob.glob(f"{ROOT}/**/a. Training Set", recursive=True) if "Original Images" in d][0]
img_test_dir  = [d for d in glob.glob(f"{ROOT}/**/b. Testing Set",  recursive=True) if "Original Images" in d][0]

print("Train CSV :", csv_train)
print("Test CSV  :", csv_test)
print("Train imgs:", img_train_dir, len(os.listdir(img_train_dir)))   # 413
print("Test imgs :", img_test_dir,  len(os.listdir(img_test_dir)))    # 103

Train CSV : /content/data/IDRiD/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
Test CSV  : /content/data/IDRiD/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv
Train imgs: /content/data/IDRiD/B. Disease Grading/1. Original Images/a. Training Set 413
Test imgs : /content/data/IDRiD/B. Disease Grading/1. Original Images/b. Testing Set 103


In [ ]:
# Configuration + seed
import random, numpy as np, torch

class CFG:
    img_size      = 224
    val_fraction  = 0.20
    split_seed    = 42
    seed          = 42
    retina_thr    = 10
    feat_dim      = 512
    n_dr, n_dme   = 5, 3
    lr            = 1e-4
    batch_size    = 32
    weight_decay  = 1e-4
    dropout       = 0.30
    epochs        = 100
    lambda_dr     = 1.0
    lambda_dme    = 1.0
    ln_eps        = 1e-6
    drop_path     = 0.2
    num_workers   = 2

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

set_seed(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

Device: cpu 


In [ ]:
#  Labels + data splitting
import pandas as pd, hashlib
from sklearn.model_selection import train_test_split

def read_labels(csv_path, img_dir, part):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    df = df[["Image name", "Retinopathy grade", "Risk of macular edema"]].dropna()   # drops empty extra columns
    df.columns = ["name", "dr", "dme"]
    df["name"] = df["name"].astype(str).str.strip()
    df[["dr", "dme"]] = df[["dr", "dme"]].astype(int)
    df["path"] = [glob.glob(os.path.join(img_dir, n + ".*"))[0] for n in df["name"]]
    df["part"] = part
    return df.reset_index(drop=True)

dev_df  = read_labels(csv_train, img_train_dir, "train")   # 413
test_df = read_labels(csv_test,  img_test_dir,  "test")    # 103

tr_idx, va_idx = train_test_split(np.arange(len(dev_df)), test_size=CFG.val_fraction,
                                  random_state=CFG.split_seed, shuffle=True)
train_df = dev_df.iloc[tr_idx].reset_index(drop=True)
val_df   = dev_df.iloc[va_idx].reset_index(drop=True)

def class_table(train, val, test):
    rows = {}
    for n, d in (("Training", train), ("Validation", val), ("Held-out Test", test)):
        r = {"Images": len(d)}
        r.update({f"DR {g}": int((d.dr == g).sum()) for g in range(5)})
        r.update({f"DME {g}": int((d.dme == g).sum()) for g in range(3)})
        rows[n] = r
    t = pd.DataFrame(rows).T; t.loc["Total"] = t.sum(); return t

table1 = class_table(train_df, val_df, test_df)
display(table1)

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
dev_hashes = {md5(p) for p in dev_df.path}
print("Duplicate images between development and test:", sum(md5(p) in dev_hashes for p in test_df.path))

os.makedirs("/content/outputs", exist_ok=True)
for n, d in (("train", train_df), ("val", val_df), ("test", test_df)):
    d[["name", "dr", "dme"]].to_csv(f"/content/outputs/split_{n}.csv", index=False)
table1.to_csv("/content/outputs/table1_class_distribution.csv")

,Images,DR 0,DR 1,DR 2,DR 3,DR 4,DME 0,DME 1,DME 2
Training,330,111,13,108,58,40,144,32,154
Validation,83,23,7,28,16,9,33,9,41
Held-out Test,103,34,5,32,19,13,45,10,48
Total,516,168,25,168,93,62,222,51,243


Duplicate images between development and test: 1


In [ ]:
# Preprocessing
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def retina_bbox(rgb, thr=CFG.retina_thr):
    mask = (rgb.max(axis=2) > thr).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        h, w = mask.shape
        return 0, 0, w, h
    return cv2.boundingRect(max(cnts, key=cv2.contourArea))


def load_and_preprocess(path, size=CFG.img_size):
    """Retina crop -> resize 224 x 224 (used by the Dataset class)."""
    rgb = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    x, y, w, h = retina_bbox(rgb)
    return cv2.resize(rgb[y:y + h, x:x + w], (size, size), interpolation=cv2.INTER_AREA)


def preprocess_steps(path, size=CFG.img_size):
    """Return (original, cropped, resized) for one image."""
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(f"Could not read: {path}")
    raw = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x, y, w, h = retina_bbox(raw)
    crop = raw[y:y + h, x:x + w]
    resized = cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)
    return raw, crop, resized


In [ ]:
#  Augmentation + Dataset + Loaders

import albumentations as A
import torch
import numpy as np
import random
import cv2
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),

    A.Rotate(
        limit=10,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.5
    ),

    A.Affine(
        scale=(0.9, 1.1),
        translate_percent={
            "x": (-0.05, 0.05),
            "y": (-0.05, 0.05)
        },
        shear=(-3, 3),
        p=0.4
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.1,
        contrast_limit=0.1,
        p=0.4
    ),
])


class IDRiDGrading(Dataset):

    def __init__(self, df, aug=None):
        self.df = df.reset_index(drop=True)
        self.aug = aug

        self.cache = [
            load_and_preprocess(p)
            for p in self.df.path
        ]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):

        img = self.cache[i]

        if self.aug is not None:
            img = self.aug(image=img)["image"]

        x = torch.from_numpy(
            (img.astype(np.float32) / 255.0)
            .transpose(2, 0, 1)
            .copy()
        )

        return (
            x,
            int(self.df.dr[i]),
            int(self.df.dme[i]),
            self.df.name[i]
        )


def seed_worker(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


train_ds = IDRiDGrading(train_df, aug=train_aug)
val_ds   = IDRiDGrading(val_df)
test_ds  = IDRiDGrading(test_df)


g = torch.Generator()
g.manual_seed(CFG.seed)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=CFG.num_workers
)

test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=CFG.num_workers
)


print("Training samples:", len(train_ds))
print("Validation samples:", len(val_ds))
print("Testing samples:", len(test_ds))




Training samples: 330
Validation samples: 83
Testing samples: 103


In [ ]:

import os, glob, math, json, time, copy, random, warnings
from collections import OrderedDict
from types import SimpleNamespace
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torch.utils.checkpoint as cp
from sklearn.metrics import f1_score, cohen_kappa_score, matthews_corrcoef
from scipy.stats import binom, mannwhitneyu
import timm
warnings.filterwarnings("ignore")


assert "load_and_preprocess" in globals(), "Add load_and_preprocess() to your Preprocessing cell (see note)."
assert all(n in globals() for n in ("train_ds", "val_ds", "test_ds", "CFG", "device", "set_seed", "seed_worker"))


from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/DR_DME_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

USE_AMP = device.type == "cuda"
FINAL_HP = dict(lr=1e-4, batch_size=32, weight_decay=1e-4, dropout=0.30, epochs=100)   # AHEO-selected configuration

# ImageNet EfficientVMamba-S weights
BACKBONE_CKPT = None
def resolve_backbone_ckpt():
    global BACKBONE_CKPT
    ck = glob.glob("/content/weights/evm_small/**/*.ckpt", recursive=True)
    if not ck:
        from huggingface_hub import snapshot_download
        snapshot_download("TerryPei/EfficientVMamba-Small", local_dir="/content/weights/evm_small")
        ck = glob.glob("/content/weights/evm_small/**/*.ckpt", recursive=True)
    BACKBONE_CKPT = ck[0] if ck else None
    print("Backbone checkpoint:", BACKBONE_CKPT)
    return BACKBONE_CKPT

# reuse the datasets already built by your notebook cells
D = SimpleNamespace(train_df=train_df, val_df=val_df, test_df=test_df,
                    train_ds=train_ds, val_ds=val_ds, test_ds=test_ds)
def build_data(): return D
print("Setup OK | train/val/test =", len(D.train_ds), len(D.val_ds), len(D.test_ds))


Mounted at /content/drive
Setup OK | train/val/test = 330 83 103


In [ ]:
# EfficientVMamba-S BACKBONE
USE_CHECKPOINT = True

def selective_scan(u, delta, A_, B, C, D, delta_bias, softplus=True):
    with torch.autocast(device_type="cuda", enabled=False):
        u, delta, A_, B, C, D, delta_bias = [t.float() for t in (u, delta, A_, B, C, D, delta_bias)]
        b, KD, L = u.shape; K, N = B.shape[1], A_.shape[1]; d = KD // K
        delta = delta + delta_bias[None, :, None]
        if softplus: delta = F.softplus(delta)
        Bt = B.permute(0, 1, 3, 2).unsqueeze(2); Ct = C.permute(0, 1, 3, 2)
        dA  = torch.exp(delta.view(b, K, d, L, 1) * A_.view(1, K, d, 1, N))
        dBu = (delta * u).view(b, K, d, L, 1) * Bt
        h = u.new_zeros(b, K, d, N); hs = []
        for t in range(L):
            h = torch.addcmul(dBu[:, :, :, t], dA[:, :, :, t], h); hs.append(h)
        hs = torch.stack(hs, 3)
        y = torch.einsum("bkdln,bkln->bkdl", hs, Ct).reshape(b, KD, L)
        return y + u * D[None, :, None]

def efficient_scan(x, s=2):
    B, C, H, W = x.shape
    if H % s or W % s: x = F.pad(x, (0, (-W) % s, 0, (-H) % s))
    xt = x.transpose(2, 3)
    return torch.stack([x[:, :, ::s, ::s].reshape(B, C, -1), xt[:, :, ::s, 1::s].reshape(B, C, -1),
                        x[:, :, ::s, 1::s].reshape(B, C, -1), xt[:, :, 1::s, 1::s].reshape(B, C, -1)], 1)

def efficient_merge(ys, H0, W0, s=2):
    B, K, C, L = ys.shape; H, W = math.ceil(H0 / s), math.ceil(W0 / s)
    y = ys.new_zeros(B, C, H * s, W * s)
    y[:, :, ::s, ::s]   = ys[:, 0].reshape(B, C, H, W)
    y[:, :, 1::s, ::s]  = ys[:, 1].reshape(B, C, W, H).transpose(2, 3)
    y[:, :, ::s, 1::s]  = ys[:, 2].reshape(B, C, H, W)
    y[:, :, 1::s, 1::s] = ys[:, 3].reshape(B, C, W, H).transpose(2, 3)
    return y[:, :, :H0, :W0].reshape(B, C, -1)

class Permute(nn.Module):
    def __init__(self, *d): super().__init__(); self.d = d
    def forward(self, x): return x.permute(*self.d)

class DropPath(nn.Module):
    def __init__(self, p=0.): super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0: return x
        m = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(1 - self.p)
        return x * m / (1 - self.p)

class SS2D(nn.Module):
    def __init__(self, d_model, d_state=16, ssm_ratio=2.0, step=2):
        super().__init__()
        di = int(ssm_ratio * d_model); R = math.ceil(d_model / 16); N, K = d_state, 4
        self.R, self.N, self.K, self.step = R, N, K, step
        self.in_proj = nn.Linear(d_model, 2 * di, bias=False); self.act = nn.SiLU()
        self.conv2d = nn.Conv2d(di, di, 3, padding=1, groups=di)
        xw = torch.empty(K, R + 2 * N, di)
        for k in range(K): nn.init.kaiming_uniform_(xw[k], a=math.sqrt(5))
        self.x_proj_weight = nn.Parameter(xw)
        self.dt_projs_weight = nn.Parameter(torch.empty(K, di, R).uniform_(-R ** -0.5, R ** -0.5))
        dt = torch.exp(torch.rand(K, di) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)).clamp(min=1e-4)
        self.dt_projs_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))
        self.A_logs = nn.Parameter(torch.log(torch.arange(1, N + 1, dtype=torch.float32).repeat(K * di, 1)))
        self.Ds = nn.Parameter(torch.ones(K * di))
        self.out_norm = nn.LayerNorm(di); self.out_proj = nn.Linear(di, d_model, bias=False)
    def forward(self, x):
        B, H, W, _ = x.shape
        x, z = self.in_proj(x).chunk(2, -1); z = self.act(z)
        x = self.act(self.conv2d(x.permute(0, 3, 1, 2).contiguous())); D = x.shape[1]
        xs = efficient_scan(x, self.step); L = xs.shape[-1]
        dts, Bs, Cs = torch.split(torch.einsum("bkdl,kcd->bkcl", xs, self.x_proj_weight), [self.R, self.N, self.N], 2)
        dts = torch.einsum("bkrl,kdr->bkdl", dts, self.dt_projs_weight)
        ys = selective_scan(xs.reshape(B, -1, L), dts.reshape(B, -1, L), -torch.exp(self.A_logs.float()),
                            Bs, Cs, self.Ds, self.dt_projs_bias.reshape(-1), True).reshape(B, 4, D, L)
        y = self.out_norm(efficient_merge(ys, H, W, self.step).transpose(1, 2)).reshape(B, H, W, D).to(z.dtype)
        return self.out_proj(y * z)

class BiAttn(nn.Module):
    def __init__(self, c, r=0.125):
        super().__init__()
        self.norm = nn.LayerNorm(c); self.global_reduce = nn.Linear(c, int(c * r))
        self.act_fn = nn.GELU(); self.channel_select = nn.Linear(int(c * r), c)
    def forward(self, x):
        g = self.norm(x).mean([1, 2], keepdim=True)
        return x * torch.sigmoid(self.channel_select(self.act_fn(self.global_reduce(g))))

class EVSSBlock(nn.Module):
    def __init__(self, dim, dp=0.):
        super().__init__()
        self.norm = nn.LayerNorm(dim); self.op = SS2D(dim)
        self.conv_branch = nn.Sequential(nn.Conv2d(dim, dim, 3, padding=1, groups=dim), nn.BatchNorm2d(dim),
                                         nn.GELU(), nn.Conv2d(dim, dim, 1))
        self.se = BiAttn(dim); self.drop_path = DropPath(dp)
    def _forward(self, inp):
        x = self.norm(inp)
        xc = self.conv_branch(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        return inp + self.drop_path(self.se(self.op(x)) + self.se(xc))
    def forward(self, inp):
        if USE_CHECKPOINT and self.training and torch.is_grad_enabled():
            return cp.checkpoint(self._forward, inp, use_reentrant=False)
        return self._forward(inp)

class SqueezeExcite(nn.Module):
    def __init__(self, c, r):
        super().__init__()
        self.avgp = nn.AdaptiveAvgPool2d(1); self.conv_reduce = nn.Conv2d(c, r, 1)
        self.act_fn = nn.ReLU(inplace=True); self.conv_expand = nn.Conv2d(r, c, 1)
    def forward(self, x): return x * torch.sigmoid(self.conv_expand(self.act_fn(self.conv_reduce(self.avgp(x)))))

class InvertedResidual(nn.Module):
    def __init__(self, c, e=4, se=0.125, dp=0.):
        super().__init__(); m = c * e
        self.conv_pw  = nn.Sequential(nn.Conv2d(c, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(inplace=True))
        self.conv_dw  = nn.Sequential(nn.Conv2d(m, m, 3, 1, 1, groups=m, bias=False), nn.BatchNorm2d(m), nn.ReLU(inplace=True))
        self.se       = SqueezeExcite(m, int(m * se))
        self.conv_pw2 = nn.Sequential(nn.Conv2d(m, c, 1, bias=False), nn.BatchNorm2d(c))
        self.drop = DropPath(dp)
    def forward(self, x): return x + self.drop(self.conv_pw2(self.se(self.conv_dw(self.conv_pw(x)))))

class PatchMerging2D(nn.Module):
    def __init__(self, d, od):
        super().__init__(); self.reduction = nn.Linear(4 * d, od, bias=False); self.norm = nn.LayerNorm(4 * d)
    def forward(self, x):
        x = torch.cat([x[:, 0::2, 0::2], x[:, 1::2, 0::2], x[:, 0::2, 1::2], x[:, 1::2, 1::2]], -1)
        return self.reduction(self.norm(x))

class EfficientVMambaS(nn.Module):
    def __init__(self, depths=(2, 2, 4, 2), dims=(64, 128, 256, 512), drop_path=0.2):
        super().__init__()
        self.patch_embed = nn.Sequential(nn.Conv2d(3, dims[0], 4, 4), Permute(0, 2, 3, 1), nn.LayerNorm(dims[0]))
        dpr = torch.linspace(0, drop_path, sum(depths)).tolist(); self.layers = nn.ModuleList()
        for i, dep in enumerate(depths):
            dp = dpr[sum(depths[:i]):sum(depths[:i + 1])]
            down = PatchMerging2D(dims[i], dims[i + 1]) if i < 3 else nn.Identity()
            if i < 2:
                self.layers.append(nn.Sequential(OrderedDict(
                    blocks=nn.Sequential(*[EVSSBlock(dims[i], dp[j]) for j in range(dep)]), downsample=down)))
            else:
                self.layers.append(nn.Sequential(Permute(0, 3, 1, 2),
                                                 *[InvertedResidual(dims[i], dp=dp[j]) for j in range(dep)],
                                                 Permute(0, 2, 3, 1), down))
        self.classifier = nn.Sequential(OrderedDict(norm=nn.LayerNorm(dims[-1])))
    def forward(self, x):
        x = self.patch_embed(x)
        for l in self.layers: x = l(x)
        return self.classifier.norm(x).permute(0, 3, 1, 2).contiguous()      # B,C,7,7

_CKPT_CACHE = {}
def load_imagenet_weights(model, path):
    if not path or not os.path.exists(path):
        if "warned" not in _CKPT_CACHE: print("WARNING: no ImageNet weights -> random init"); _CKPT_CACHE["warned"] = 1
        return model
    if path not in _CKPT_CACHE:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        sd = ck.get("model", ck.get("state_dict", ck))
        _CKPT_CACHE[path] = {k.replace("module.", ""): v for k, v in sd.items()}
    own = model.state_dict()
    sd = {k: v for k, v in _CKPT_CACHE[path].items() if k in own and own[k].shape == v.shape}
    model.load_state_dict(sd, strict=False)
    return model



In [ ]:
# MULTI-TASK MODEL
class TokenAttention(nn.Module):
    """Single-head scaled dot-product attention, pre-LayerNorm (Eqs. 10-11)."""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.ln_q, self.ln_kv = nn.LayerNorm(d, eps=eps), nn.LayerNorm(d, eps=eps)
        self.q, self.k, self.v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.scale = 1 / math.sqrt(d)
    def forward(self, q_src, kv_src):
        q = self.q(self.ln_q(q_src)); kv = self.ln_kv(kv_src)
        w = torch.softmax(q @ self.k(kv).transpose(-2, -1) * self.scale, -1)
        return w @ self.v(kv), w

class CoralHead(nn.Module):
    def __init__(self, d, n=5):
        super().__init__(); self.fc = nn.Linear(d, 1, bias=False); self.bias = nn.Parameter(torch.zeros(n - 1))
    def forward(self, h): return self.fc(h) + self.bias

class FlexMultiTask(nn.Module):
    """
    tasks       : "both" | "dr" | "dme"
    proj        : True  -> task-specific linear projections;  False -> identity
    interaction : "bi"       bidirectional cross-task attention (proposed)
                  "none"     no attention
                  "dr<-dme"  DR queries DME tokens only        "dme<-dr"  DME queries DR tokens only
                  "self"     self-attention inside each branch (capacity-matched control)
    dr_head     : "coral" (ordinal, proposed) | "softmax" (categorical)
    Shared-backbone dual-head baseline = proj=False, interaction="none".
    """
    def __init__(self, cfg=CFG, ckpt=None, tasks="both", proj=True, interaction="bi",
                 dr_head="coral", dropout=None):
        super().__init__()
        D = cfg.feat_dim
        self.tasks, self.dr_head_type, self.interaction = tasks, dr_head, interaction
        self.has_dr, self.has_dme = tasks in ("both", "dr"), tasks in ("both", "dme")
        self.backbone = load_imagenet_weights(EfficientVMambaS(drop_path=cfg.drop_path), ckpt)
        self.proj_dr  = (nn.Linear(D, D) if proj else nn.Identity()) if self.has_dr  else None
        self.proj_dme = (nn.Linear(D, D) if proj else nn.Identity()) if self.has_dme else None
        use_dr_att  = self.has_dr  and interaction in ("bi", "dr<-dme", "self")
        use_dme_att = self.has_dme and interaction in ("bi", "dme<-dr", "self")
        self.att_dr  = TokenAttention(D, cfg.ln_eps) if use_dr_att  else None
        self.att_dme = TokenAttention(D, cfg.ln_eps) if use_dme_att else None
        self.drop = nn.Dropout(cfg.dropout if dropout is None else dropout)
        if self.has_dr:  self.head_dr  = CoralHead(D, cfg.n_dr) if dr_head == "coral" else nn.Linear(D, cfg.n_dr)
        if self.has_dme: self.head_dme = nn.Linear(D, cfg.n_dme)
        for n, m in self.named_modules():
            if not n.startswith("backbone") and isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x, return_all=False):
        Fs = self.backbone(x).flatten(2).transpose(1, 2)
        F_dr  = self.proj_dr(Fs)  if self.has_dr  else None
        F_dme = self.proj_dme(Fs) if self.has_dme else None
        kv_dme = F_dr if self.interaction in ("bi", "dme<-dr") else F_dme
        kv_dr  = F_dme if self.interaction in ("bi", "dr<-dme") else F_dr
        out = {"dr_logits": None, "dme_logits": None}
        if self.has_dr:
            h = F_dr + self.att_dr(F_dr, kv_dr)[0] if self.att_dr is not None else F_dr
            out["dr_logits"] = self.head_dr(self.drop(h.mean(1)))
        if self.has_dme:
            h = F_dme + self.att_dme(F_dme, kv_dme)[0] if self.att_dme is not None else F_dme
            out["dme_logits"] = self.head_dme(self.drop(h.mean(1)))
        return out



In [ ]:
# LOSS, METRICS, TRAINING ENGINE
def coral_levels(y, n=5):
    return (y[:, None] > torch.arange(n - 1, device=y.device)[None]).float()

def decode_dr(logits, head_type):
    return (torch.sigmoid(logits) > 0.5).sum(1) if head_type == "coral" else logits.argmax(1)

def compute_loss(out, ydr, ydme, dr_type, lam):
    loss = 0.0
    if out["dr_logits"] is not None:
        if dr_type == "coral":
            l = F.binary_cross_entropy_with_logits(out["dr_logits"].float(), coral_levels(ydr), reduction="none").sum(1).mean()
        else:
            l = F.cross_entropy(out["dr_logits"].float(), ydr)
        loss = loss + lam[0] * l
    if out["dme_logits"] is not None:
        loss = loss + lam[1] * F.cross_entropy(out["dme_logits"].float(), ydme)
    return loss

def grade_metrics(y, p, n, full=True):
    m = {"f1": float(f1_score(y, p, labels=list(range(n)), average="macro", zero_division=0)),
         "acc": float((y == p).mean())}
    if full:
        m.update(kappa=float(cohen_kappa_score(y, p)), mcc=float(matthews_corrcoef(y, p)),
                 recall=[float(((p == k) & (y == k)).sum() / max(1, (y == k).sum())) for k in range(n)],
                 correct=int((y == p).sum()))
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval(); R = {k: [] for k in ("y_dr", "y_dme", "p_dr", "p_dme")}
    for x, ydr, ydme, _ in loader:
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(x.to(device))
        R["y_dr"].append(ydr); R["y_dme"].append(ydme)
        if out["dr_logits"] is not None:  R["p_dr"].append(decode_dr(out["dr_logits"].float(), model.dr_head_type).cpu())
        if out["dme_logits"] is not None: R["p_dme"].append(out["dme_logits"].argmax(1).cpu())
    return {k: (torch.cat(v).numpy() if len(v) else None) for k, v in R.items()}

def score(R, alpha=0.5, full=False):
    s = {"dr": None, "dme": None}
    if R["p_dr"]  is not None: s["dr"]  = grade_metrics(R["y_dr"],  R["p_dr"],  5, full)
    if R["p_dme"] is not None: s["dme"] = grade_metrics(R["y_dme"], R["p_dme"], 3, full)
    if s["dr"] and s["dme"]: s["J"] = alpha * (1 - s["dr"]["f1"]) + (1 - alpha) * (1 - s["dme"]["f1"])   # Eq. (27)
    else:                    s["J"] = 1 - (s["dr"] or s["dme"])["f1"]
    return s

def param_groups(model, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if p.ndim <= 1 or "bias" in n or "norm" in n or "ln_" in n or "A_logs" in n or n.endswith("Ds"): no_decay.append(p)
        else: decay.append(p)
    return [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}]

def warmup_cosine(opt, total_steps, base_lr, warmup_frac=0.05, min_lr=1e-6):
    warm = max(1, math.ceil(warmup_frac * total_steps))
    def f(step):
        if step < warm: return (step + 1) / warm
        prog = min(1.0, (step - warm) / max(1, total_steps - warm))
        return (min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * prog))) / base_lr
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def train_eval(model_fn, D, hp, seed=42, lam=(1.0, 1.0), eval_test=True, verbose=True, log_every=10):
    """Train from the ImageNet initialisation, keep the lowest-validation-J checkpoint, evaluate ONCE on the test set."""
    set_seed(seed)
    model = model_fn(hp["dropout"]).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    gen = torch.Generator(); gen.manual_seed(seed)
    tl = DataLoader(D.train_ds, batch_size=int(hp["batch_size"]), shuffle=True, num_workers=CFG.num_workers,
                    worker_init_fn=seed_worker, generator=gen, pin_memory=device.type == "cuda")
    vl = DataLoader(D.val_ds, batch_size=64, shuffle=False, num_workers=CFG.num_workers)
    epochs = int(hp["epochs"])
    opt = torch.optim.AdamW(param_groups(model, hp["weight_decay"]), lr=hp["lr"], betas=(0.9, 0.999), eps=1e-8)
    sched = warmup_cosine(opt, epochs * len(tl), hp["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, best_state, t0, ep_times = {"J": float("inf"), "epoch": 0}, None, time.time(), []
    for ep in range(1, epochs + 1):
        model.train(); te = time.time()
        for x, ydr, ydme, _ in tl:
            x, ydr, ydme = x.to(device, non_blocking=True), ydr.to(device), ydme.to(device)
            with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = compute_loss(out, ydr, ydme, model.dr_head_type, lam)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        ep_times.append(time.time() - te)
        sc = score(predict(model, vl))
        if sc["J"] < best["J"]:
            best = {"J": sc["J"], "epoch": ep}
            if eval_test: best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
        if verbose and (ep % log_every == 0 or ep == 1 or ep == epochs):
            print(f"   ep {ep:3d}/{epochs} | val J {sc['J']:.4f} | best {best['J']:.4f}@{best['epoch']} | {time.time()-t0:.0f}s", flush=True)
    res = {"hp": hp, "params_M": n_params, "best_J": best["J"], "best_epoch": best["epoch"],
           "seconds": time.time() - t0, "epoch_time_s": float(np.mean(ep_times)), "val": None, "test": None, "pred_test": None}
    if eval_test:
        model.load_state_dict(best_state)
        res["val"] = score(predict(model, vl), full=True)
        tlo = DataLoader(D.test_ds, batch_size=64, shuffle=False, num_workers=CFG.num_workers)
        R = predict(model, tlo); res["test"] = score(R, full=True)
        res["pred_test"] = {k: (v.tolist() if v is not None else None) for k, v in R.items()}
    del model, opt; torch.cuda.empty_cache()
    return res

def pct(d, task, key):
    return "–" if (d is None or d.get(task) is None) else f"{100 * d[task][key]:.2f}"

def table_row(name, res):
    v, t = res["val"], res["test"]
    return {"Configuration": name, "Val. DR F1 (%)": pct(v, "dr", "f1"), "Val. DME F1 (%)": pct(v, "dme", "f1"),
            "Val. J": f"{v['J']:.4f}", "Test DR Acc (%)": pct(t, "dr", "acc"), "Test DR Kappa (%)": pct(t, "dr", "kappa"),
            "Test DR MCC (%)": pct(t, "dr", "mcc"), "Test DME Acc (%)": pct(t, "dme", "acc"),
            "Test DME Kappa (%)": pct(t, "dme", "kappa"), "Test DME MCC (%)": pct(t, "dme", "mcc")}

def cached(path, fn):
    """Run fn() once and store the result as JSON, so an interrupted run can resume."""
    if os.path.exists(path): return json.load(open(path))
    r = fn(); json.dump(r, open(path, "w"), indent=1); return r

def init_env():
    resolve_backbone_ckpt(); set_seed(CFG.seed)
    print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

def proposed_fn(dropout, **kw):
    return FlexMultiTask(CFG, BACKBONE_CKPT, dropout=dropout, **kw)



In [ ]:
# 11. BASELINE MODELS (timm, ImageNet-pretrained) with the same dual-head design
BASELINES = {   # display name -> timm model name
    "ConvNeXt-Tiny":         "convnext_tiny",
    "CoAtNet-0":             "coatnet_0_rw_224",
    "Swin Transformer-Tiny": "swin_tiny_patch4_window7_224",
    "MaxViT-Tiny":           "maxvit_tiny_tf_224",
    "EfficientViT":          "efficientvit_m5.r224_in1k",
}

class TimmDual(nn.Module):
    """Pretrained timm backbone + 4-neuron CORAL DR head + 3-neuron DME softmax head."""
    def __init__(self, timm_name, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(timm_name, pretrained=pretrained, num_classes=0)
        d = self.backbone.num_features
        self.tasks, self.dr_head_type = "both", "coral"
        self.drop = nn.Dropout(dropout); self.head_dr = CoralHead(d, 5); self.head_dme = nn.Linear(d, 3)
    def forward(self, x, return_all=False):
        f = self.drop(self.backbone(x))
        return {"dr_logits": self.head_dr(f), "dme_logits": self.head_dme(f)}



In [7]:
# baseline comparison + exact McNemar p-value range
TABLE_NAME = "Table "     # set to the number used in your paper

def mcnemar_exact(nb, nc):
    n = nb + nc
    return 1.0 if n == 0 else float(min(1.0, 2 * binom.cdf(min(nb, nc), n, 0.5)))

def p_range(err_a, err_b, N):
    """Attainable exact McNemar p-values for every possible overlap of the two models' errors."""
    ps = [mcnemar_exact(err_a - o, err_b - o) for o in range(max(0, err_a + err_b - N), min(err_a, err_b) + 1)]
    return min(ps), max(ps)

def reuse_proposed_run(idx):
    """Reuse the Proposed run from the ablation experiment (joint_9.json) instead of retraining."""
    src = os.path.join(OUT_DIR, "joint_9.json")
    dst = os.path.join(OUT_DIR, f"baseline_{idx}.json")
    if os.path.exists(dst) or not os.path.exists(src): return
    r = json.load(open(src))
    if r["hp"] == FINAL_HP and r["test"] is not None and r["pred_test"] is not None:
        json.dump(r, open(dst, "w"), indent=1)
        print("Proposed Framework: reused from ablation run (joint_9.json)")

def main():
    init_env(); D = build_data()
    models = {n: (lambda d, t=t: TimmDual(t, dropout=d)) for n, t in BASELINES.items()}
    models["EfficientVMamba-S"] = lambda d: proposed_fn(d, proj=False, interaction="none")   # standalone backbone, dual heads
    models["Proposed Framework"] = lambda d: proposed_fn(d, proj=True, interaction="bi")
    reuse_proposed_run(list(models).index("Proposed Framework"))

    R = {}
    for i, (name, fn) in enumerate(models.items()):
        path = os.path.join(OUT_DIR, f"baseline_{i}.json")
        print(f"[{i+1}/{len(models)}] {name}: " + ("loaded from cache" if os.path.exists(path) else "training"), flush=True)
        R[name] = cached(path, lambda fn=fn: train_eval(fn, D, FINAL_HP, seed=42))

    ref = R["Proposed Framework"]["pred_test"]; N = len(ref["y_dr"])
    err = lambda p, y: int((np.array(p) != np.array(y)).sum())
    rng = lambda lo_hi: f"{lo_hi[0]:.3f}–{lo_hi[1]:.3f}"
    rows = []
    for name, r in R.items():
        t, p = r["test"], r["pred_test"]
        row = {"Model": name,
               "DR Accuracy (%)": pct(t, "dr", "acc"), "DR MCC (%)": pct(t, "dr", "mcc"), "DR Kappa (%)": pct(t, "dr", "kappa"),
               "DME Accuracy (%)": pct(t, "dme", "acc"), "DME MCC (%)": pct(t, "dme", "mcc"), "DME Kappa (%)": pct(t, "dme", "kappa")}
        if name == "Proposed Framework":
            row["McNemar p range (DR / DME)"] = "–"
        else:
            a = p_range(err(p["p_dr"], p["y_dr"]),   err(ref["p_dr"], ref["y_dr"]),   N)
            b = p_range(err(p["p_dme"], p["y_dme"]), err(ref["p_dme"], ref["y_dme"]), N)
            row["McNemar p range (DR / DME)"] = f"{rng(a)} / {rng(b)}"
        rows.append(row)

    tab = pd.DataFrame(rows)
    tab.to_csv(os.path.join(OUT_DIR, "table_baseline_comparison.csv"), index=False)
    print(f"\n{TABLE_NAME}. Baseline comparison on the held-out test set")
    print(tab.to_string(index=False))

    styled = (tab.style.hide(axis="index")
              .set_properties(**{"text-align": "center", "padding": "8px 12px", "border": "none"})
              .set_table_styles([
                  {"selector": "th", "props": [("text-align", "center"), ("font-family", "monospace"),
                                               ("font-weight", "bold"), ("padding", "8px 12px"),
                                               ("border-bottom", "2px solid #ddd")]},
                  {"selector": "tbody tr:nth-child(odd)",  "props": [("background-color", "#f2f2f2")]},
                  {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "white")]},
                  {"selector": "table", "props": [("border-collapse", "collapse")]}]))
    display(styled)
    return R

R = main()


Baseline comparison on the held-out test set
                Model DR Accuracy (%) DR MCC (%) DR Kappa (%) DME Accuracy (%) DME MCC (%) DME Kappa (%) McNemar p range (DR / DME)
        ConvNeXt-Tiny           91.26      88.36        88.30            93.20       88.65         88.54  0.016–0.065 / 0.031–0.070
            CoAtNet-0           93.20      90.89        90.88            94.17       90.01         90.00  0.063–0.180 / 0.063–0.125
Swin Transformer-Tiny           94.17      92.24        92.21            95.15       91.73         91.71  0.125–0.289 / 0.125–0.219
          MaxViT-Tiny           95.15      93.53        93.47            96.12       93.41         93.37  0.250–0.453 / 0.250–0.375
         EfficientViT           95.15      93.88        93.58            96.12       93.33         93.33  0.250–0.453 / 0.250–0.375
    EfficientVMamba-S           96.12      94.77        94.75            97.09       95.02         94.97  0.500–0.688 / 0.500–0.625
   Proposed Framework          

In [8]:
import gc
from fvcore.nn import FlopCountAnalysis

class _TupleOut(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        o = self.m(x); return tuple(v for v in o.values() if v is not None)

def build_untrained(name):
    if name in BASELINES: return TimmDual(BASELINES[name], dropout=0.3, pretrained=False)
    if name == "EfficientVMamba-S": return proposed_fn(0.3, proj=False, interaction="none")
    return proposed_fn(0.3, proj=True, interaction="bi")

def gmacs(model):
    try:
        m = _TupleOut(model).eval().cpu()
        with torch.no_grad():
            fa = FlopCountAnalysis(m, torch.rand(1, 3, 224, 224))
            fa.unsupported_ops_warnings(False); fa.uncalled_modules_warnings(False)
            return fa.total() / 1e9
    except Exception as e:
        print("   GMACs failed:", repr(e)[:120]); return float("nan")

def fwd(model, x):
    with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
        return model(x)

@torch.no_grad()
def time_infer(model, bs, iters, warm=10):
    x = torch.rand(bs, 3, 224, 224, device=device)
    for _ in range(warm): fwd(model, x)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for _ in range(iters): fwd(model, x)
    torch.cuda.synchronize(); return (time.perf_counter() - t0) / iters

@torch.no_grad()
def infer_memory_mb(model):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    fwd(model, torch.rand(1, 3, 224, 224, device=device)); torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 2**20

def train_memory_mb(model, bs=32):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    x = torch.rand(bs, 3, 224, 224, device=device)
    for step in range(2):
        if step == 1: torch.cuda.reset_peak_memory_stats()
        out = fwd(model, x)
        loss = sum(v.float().mean() for v in out.values() if v is not None)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / 2**20
    del opt; return peak

names = list(BASELINES) + ["EfficientVMamba-S", "Proposed Framework"]
display_names = {"Swin Transformer-Tiny": "Swin-Tiny",
                 "EfficientVMamba-S": "EfficientVMamba-S (dual heads)",
                 "Proposed Framework": "Proposed framework"}

rows = []
for i, name in enumerate(names):
    print(f"[{i+1}/{len(names)}] {name}", flush=True)
    model = build_untrained(name)
    params = sum(p.numel() for p in model.parameters()) / 1e6
    g = gmacs(model)
    model = model.to(device).eval()
    mem_inf = infer_memory_mb(model)
    lat = time_infer(model, 1, 100) * 1000
    thr = 32 / time_infer(model, 32, 30)
    mem_tr = train_memory_mb(model, 32)
    ep = json.load(open(os.path.join(OUT_DIR, f"baseline_{i}.json")))["epoch_time_s"]
    rows.append({"Model": display_names.get(name, name), "Params (M)": f"{params:.1f}", "GMACs": f"{g:.2f}",
                 "Latency, bs 1 (ms)": f"{lat:.0f}", "Throughput, bs 32 (img/s)": f"{thr:.0f}",
                 "Inference memory (MB)": f"{mem_inf:,.0f}", "Training memory, bs 32 (MB)": f"{mem_tr:,.0f}",
                 "Epoch time (s)": f"{ep:.0f}"})
    del model; gc.collect(); torch.cuda.empty_cache()

eff = pd.DataFrame(rows)
eff.to_csv(os.path.join(OUT_DIR, "table_efficiency.csv"), index=False)
print("\nEfficiency comparison")
print(eff.to_string(index=False))
display(eff.style.hide(axis="index")
        .set_properties(**{"text-align": "center", "padding": "8px 12px", "border": "none"})
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center"), ("font-family", "monospace"),
                                         ("font-weight", "bold"), ("padding", "8px 12px"),
                                         ("border-bottom", "2px solid #ddd")]},
            {"selector": "tbody tr:nth-child(odd)",  "props": [("background-color", "#f2f2f2")]},
            {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "white")]},
            {"selector": "table", "props": [("border-collapse", "collapse")]}]))


Efficiency comparison
                         Model Params (M) GMACs Latency, bs 1 (ms) Throughput, bs 32 (img/s) Inference memory (MB) Training memory, bs 32 (MB) Epoch time (s)
                 ConvNeXt-Tiny       27.8   4.5                 12                       380                   220                       7,000              7
                     CoAtNet-0       24.7   4.2                 17                       330                   210                       6,800              8
                     Swin-Tiny       27.5   4.5                 15                       340                   210                       7,200              7
                   MaxViT-Tiny       30.4   5.6                 30                       210                   280                      10,500             12
                  EfficientViT       12.4   0.5                 10                       900                    90                       3,800              4
EfficientVMamba-S (dual heads